In [1]:
from neo4j import GraphDatabase
import dotenv
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import community as community_louvain 

In [2]:

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))


driver = GraphDatabase.driver(URI, auth=AUTH)

DIAG_LABEL = "Diagnostico"   
DIAG_NAME_PROP = "terminoEN"     
PACIENTE_LABEL = "Paciente"      

In [3]:
def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

In [12]:
# 1) ¿Hay nodos Paciente?
q_count_pat = f"MATCH (p:{PACIENTE_LABEL}) RETURN count(p) AS cnt"
cnt_pat = run_query(q_count_pat)[0]["cnt"]
print(f"Pacientes encontrados: {cnt_pat}")

# 2) Cargar nodos Diagnostico (id -> nombre)
q_nodos = f"MATCH (d:{DIAG_LABEL}) RETURN elementId(d) AS id, d.{DIAG_NAME_PROP} AS nombre"
nodos_raw = run_query(q_nodos)
id2name = {row["id"]: row["nombre"] or f"diag_{row['id']}" for row in nodos_raw}
print(f"Diagnósticos cargados: {len(id2name)}")

Pacientes encontrados: 1567
Diagnósticos cargados: 1746


In [13]:
# Construimos un grafo no dirigido, que es lo habitual para Louvain y WCC
G = nx.Graph()

for nid, nombre in id2name.items():
    G.add_node(nid, label=nombre)

if cnt_pat > 0:
    # Extraer, para cada paciente, la lista de diagnósticos conectados (cualquier tipo de relación)
    # Usamos cualquier relación entrante/saliente entre paciente y diagnostico
    q_rels = f"""
    MATCH (p:{PACIENTE_LABEL})-[r]->(d:{DIAG_LABEL})
    RETURN elementId(p) AS pid, collect(DISTINCT elementId(d)) AS diagnosticos
    """
    rels = run_query(q_rels)
    # Construir co-ocurrencia: dos diagnósticos conectados si aparecen en el mismo paciente
    for row in rels:
        diags = row["diagnosticos"] or []
        # si el paciente solo tiene 0 o 1 diagnóstico, no genera aristas
        for i in range(len(diags)):
            for j in range(i+1, len(diags)):
                a, b = diags[i], diags[j]
                if G.has_edge(a, b):
                    G[a][b]["weight"] += 1
                else:
                    G.add_edge(a, b, weight=1)
    print("Grafo de co-ocurrencia construido a partir de pacientes.")
else:
    # Fallback: extraer relaciones directas entre diagnósticos
    # Capturamos cualquier relación de Diagnostico -> Diagnostico (p. ej. DIAGNOSTICO_ASOCIADO, DIAGNOSTICOS_PSIQUIATRICO)
    q_diag_links = f"""
    MATCH (d1:{DIAG_LABEL})-[r]->(d2:{DIAG_LABEL})
    RETURN elementId(d1) AS a, elementId(d2) AS b, type(r) AS relType, count(*) AS cnt
    """
    links = run_query(q_diag_links)
    for row in links:
        a, b, cnt = row["a"], row["b"], row["cnt"]
        # Tratamos la proyección como no dirigida (si quieres dirigir, cambia a DiGraph)
        if G.has_edge(a, b):
            G[a][b]["weight"] += cnt
        else:
            G.add_edge(a, b, weight=cnt)
    print("Grafo construido a partir de relaciones directas diagnóstico→diagnóstico.")

Grafo de co-ocurrencia construido a partir de pacientes.


In [26]:
# Computar particiones
partition = community_louvain.best_partition(G)

# Convertimos a DataFrame para analizar fácilmente
louvain_df = pd.DataFrame([
    {"diagnostico": node, "comunidad": community}
    for node, community in partition.items()
])

print(louvain_df.head())

                                diagnostico  comunidad
0  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:0         16
1  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:1         16
2  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:2         16
3  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:3          1
4  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:4          7


In [27]:
num_comunidades = louvain_df["comunidad"].nunique()
print("Número de comunidades detectadas:", num_comunidades)


Número de comunidades detectadas: 49


In [28]:
components = list(nx.connected_components(G))

print("Número de componentes:", len(components))
print("Tamaño de cada componente:", [len(c) for c in components])


Número de componentes: 41
Tamaño de cada componente: [1706, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [18]:
coloring = nx.coloring.greedy_color(G, strategy="largest_first")

coloring_df = pd.DataFrame([
    {"diagnostico": node, "color": color}
    for node, color in coloring.items()
])

print(coloring_df.head())
num_colores = len(set(coloring.values()))
print("Número de colores utilizados:", num_colores)


                                  diagnostico  color
0  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:370      0
1  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:371      1
2  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:363      2
3  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:364      3
4  4:eeba0f6e-886b-4c3e-88b1-6ee081f95425:570      4
Número de colores utilizados: 68


In [19]:
from pyvis.network import Network

net = Network(notebook=True, width="1000px", height="700px")

for node, comunidad in partition.items():
    net.add_node(node, label=node, color=f"hsl({comunidad*40 % 360}, 70%, 50%)")

for u, v in G.edges():
    net.add_edge(u, v)

net.show("louvain_network.html")


KeyboardInterrupt: 